In [0]:
\++\9#/Volumes/dataengineer/ingestion/netflix
df = spark.read.option("header","true").csv("/Volumes/dataengineer/ingestion/netflix/netflix_titles.csv")
df.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan

# Import Libraries

In [0]:
from pyspark.sql.functions import *
from datetime import datetime
import hashlib
import os

#  Define File Path


In [0]:

file_path = "/Volumes/dataengineer/ingestion/netflix/netflix_titles.csv"

# Pipeline Start Time


In [0]:
pipeline_start_time = datetime.now()

print("Pipeline Start Time:", pipeline_start_time)

Pipeline Start Time: 2026-06-24 11:44:27.049599


# Source File Availability Validation


In [0]:
display(dbutils.fs.ls("/Volumes/dataengineer/ingestion/netflix/"))

path,name,size,modificationTime
dbfs:/Volumes/dataengineer/ingestion/netflix/netflix_titles.csv,netflix_titles.csv,3399671,1782299400000


# File Format Validation


In [0]:
file_format = file_path.split(".")[-1]

if file_format.lower() == "csv":
    print("Valid CSV File")
else:
    raise Exception("Invalid File Format")

Valid CSV File


# Read Source File

In [0]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(file_path)

df.show(5)

+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                NULL|United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|           NULL|Ama Qamata, Khosi...| South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglands|Julien Leclercq|Sami Bouajila, Tr...|         NULL|Septem

# File Size Validation


In [0]:
record_count = df.count()

if record_count > 0:
    print("File Contains Data")
else:
    raise Exception("Empty File")

File Contains Data


# Source File Name Capture


In [0]:
file_name = file_path.split("/")[-1]

print(file_name)

netflix_titles.csv


# Batch ID Capture


In [0]:
batch_id = "BATCH_" + datetime.now().strftime("%Y%m%d%H%M%S")

print(batch_id)

BATCH_20260624114843


# Processing Date Capture


In [0]:
processing_date = datetime.now().date()

print(processing_date)

2026-06-24


# Ingestion Timestamp Capture


In [0]:
df = df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

# Add Source File Name Column


In [0]:
df = df.withColumn(
    "source_file_name",
    lit(file_name)
)

# Add Batch ID Column


In [0]:
df = df.withColumn(
    "batch_id",
    lit(batch_id)
)

# Add Processing Date Column


In [0]:
df = df.withColumn(
    "processing_date",
    current_date()
)

# Display Data After Metadata Addition


In [0]:
df.show(5)

+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+--------------------+------------------+--------------------+---------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description| ingestion_timestamp|  source_file_name|            batch_id|processing_date|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+--------------------+------------------+--------------------+---------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                NULL|United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|2026-06-24 11:53:...|netflix_titles.csv|BATCH_

# Empty File Validation


In [0]:
if df.count() == 0:
    raise Exception("File is Empty")
else:
    print("File Passed Empty File Validation")

File Passed Empty File Validation


# Corrupt File Validation


In [0]:
try:
    df = spark.read \
        .option("header", True) \
        .csv(file_path)

    print("File is Not Corrupt")

except Exception as e:
    raise Exception(f"Corrupt File: {e}")

File is Not Corrupt


# Duplicate File Validation


In [0]:
file_hash = hashlib.md5(file_name.encode()).hexdigest()

print("File Hash:", file_hash)

File Hash: f658e279b79af433d89a861d9e5c21fe


# Schema Validation

In [0]:
expected_columns = [
    'show_id',
    'type',
    'title',
    'director',
    'cast',
    'country',
    'date_added',
    'release_year',
    'rating',
    'duration',
    'listed_in',
    'description'
]

if df.columns[:12] == expected_columns:
    print("Schema Validation Passed")
else:
    raise Exception("Schema Validation Failed")

Schema Validation Passed


# Total Records Read


In [0]:
total_records = df.count()

print("Total Records Read:", total_records)

Total Records Read: 8809


# Processing Status


In [0]:
status = "SUCCESS"

print(status)

SUCCESS


# Audit Information Generation


In [0]:
audit_data = [(
    batch_id,
    file_name,
    total_records,
    str(processing_date),
    status
)]

audit_columns = [
    "batch_id",
    "file_name",
    "record_count",
    "processing_date",
    "status"
]

audit_df = spark.createDataFrame(
    audit_data,
    audit_columns
)

audit_df.show()

+--------------------+------------------+------------+---------------+-------+
|            batch_id|         file_name|record_count|processing_date| status|
+--------------------+------------------+------------+---------------+-------+
|BATCH_20260624114843|netflix_titles.csv|        8809|     2026-06-24|SUCCESS|
+--------------------+------------------+------------+---------------+-------+



# Load Raw Data to Landing Layer

In [0]:
landing_path = "/Volumes/dataengineer/landing/netflix/"

# Pipeline End Time


In [0]:
pipeline_end_time = datetime.now()

print("Pipeline End Time:", pipeline_end_time)

Pipeline End Time: 2026-06-24 12:02:29.491961


# Execution Log


In [0]:
print("====================================")
print("PIPELINE EXECUTION LOG")
print("====================================")

print("Start Time :", pipeline_start_time)
print("End Time   :", pipeline_end_time)
print("File Name  :", file_name)
print("Batch ID   :", batch_id)
print("Records    :", total_records)
print("Status     :", status)

print("====================================")

PIPELINE EXECUTION LOG
Start Time : 2026-06-24 11:44:27.049599
End Time   : 2026-06-24 12:02:29.491961
File Name  : netflix_titles.csv
Batch ID   : BATCH_20260624114843
Records    : 8809
Status     : SUCCESS
